In [ ]:
using Pkg
Pkg.activate(expanduser("~/juliaenvs/dev"))   # point to your own environment

In [ ]:
using TinyMachines
using Flux
using CUDA
dev = CUDA.has_cuda_gpu() ? gpu : cpu

In [ ]:
# Synth data
Xs = rand(Float32, 64,64,3,10)   # 3-ch image simulation
ys = rand(Bool, 64,64,2,10)      # bool mask simulation

data = Flux.DataLoader((Xs, ys)) |> dev

In [ ]:
# model
model = UNet(3,2) |> dev;

In [ ]:
# loss
function lossfn(model, X, y)   # args as required by Flux.train!
    yhat = model(X)
    return Flux.logitcrossentropy(yhat, y, dims=3)
end

In [ ]:
# optimizer
opt = Flux.Adam()
opt_state = Flux.setup(opt, model)

In [ ]:
# train one epoch
Flux.train!(lossfn, model, data, opt_state)

In [ ]:
# inference
X = rand(Float32, 64,64,3,1) |> dev
ŷ = model(X) |> x->softmax(x,dims=3) |> cpu
@assert size(ŷ) == (64,64,2,1)
@info "pass"